# pcKeyboard — neurális reranker tréning

Futtasd a cellákat **sorban, fentről lefelé** (Shift+Enter).

Előtte: **Runtime → Change runtime type → T4 GPU**, majd Save.

A végén két fájlt kapsz (`reranker_hu_HU.tflite` és `reranker_hu_HU.chars`) —
ezeket kell a repo `app/src/main/assets/reranker/` mappájába feltölteni.

In [ ]:
# 1) GPU-ellenőrzés — a táblázatban egy Tesla T4-nek kell látszania.
!nvidia-smi
import tensorflow as tf
print('TF:', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# 2) Nyelv kiválasztása — először a magyart érdemes.
LANG = 'hu_HU'
CORPUS = 'hun_news_2020_1M'   # en_US: eng_news_2020_1M, de_DE: deu_news_2020_1M, es_ES: spa_news_2020_1M

In [ ]:
# 3) Tréning-szkriptek beszerzése (újrafuttatható — mindig friss másolatot húz).
# Ha a repo privát, a klónozás elhasal — akkor a felugró gombbal töltsd fel
# kézzel a scripts/train_reranker/ mappából: train.py, noise_model.py, evaluate.py
import os
os.system('rm -rf repo train.py noise_model.py evaluate.py')
ok = os.system('git clone --depth 1 https://github.com/9hm2/pcKeyboard.git repo') == 0
if ok:
    os.system('cp repo/scripts/train_reranker/*.py .')
if not os.path.exists('train.py'):
    from google.colab import files
    print('Töltsd fel: train.py, noise_model.py, evaluate.py')
    files.upload()
print('train.py megvan:', os.path.exists('train.py'))

In [ ]:
# 4) Korpusz letöltése (~250 MB) és kicsomagolása.
!wget -q --show-progress https://downloads.wortschatz-leipzig.de/corpora/{CORPUS}.tar.gz
!tar xzf {CORPUS}.tar.gz --wildcards '*-sentences.txt'
SENTENCES = f'{CORPUS}/{CORPUS}-sentences.txt'
!wc -l {SENTENCES}

In [ ]:
# 5) GYORS PRÓBA (~10-15 perc): kis adaton ellenőrzi, hogy a lánc végigmegy.
!python3 train.py --lang {LANG} --sentences {SENTENCES} --epochs 1 --limit-sentences 100000
!ls -la reranker_{LANG}.*

In [ ]:
# 6) TELJES TRÉNING (~2-4 óra T4-en). Hagyd futni; a loss szépen csökkenjen.
# Ha a Colab útközben lecsatlakoztat, futtasd újra ezt a cellát.
!python3 train.py --lang {LANG} --sentences {SENTENCES} --epochs 2
!ls -la reranker_{LANG}.*

In [ ]:
# 7) Kiértékelés TANÍTÁSON KÍVÜLI (held-out) mondatokon.
# Több évjáratot próbál; az első létező korpuszt használja.
import os
held = None
for cand in ['hun_news_2019_300K', 'hun_news_2018_300K', 'hun_newscrawl_2019_300K',
             'hun_wikipedia_2021_300K', 'hun_mixed_2012_300K']:
    if os.system(f'wget -q https://downloads.wortschatz-leipzig.de/corpora/{cand}.tar.gz') == 0:
        os.system(f'tar xzf {cand}.tar.gz --wildcards "*-sentences.txt"')
        held = f'{cand}/{cand}-sentences.txt'
        break
print('held-out:', held)
if held:
    !pip -q install ai-edge-litert
    !python3 evaluate.py --lang {LANG} --sentences {held} --model reranker_{LANG}.tflite --chars reranker_{LANG}.chars

**Döntés a 7) cella számai alapján:**
- `synthetic top-1` **≥ 90%** és a live-esetek mind `True` → mehet a 8) cella.
- Gyengébb eredménynél futtasd újra a 6) cellát `--epochs 3`-mal, majd megint a 7)-et.

In [ ]:
# 8) A két kimeneti fájl letöltése a gépedre.
from google.colab import files
files.download(f'reranker_{LANG}.tflite')
files.download(f'reranker_{LANG}.chars')

## Utolsó lépés — beépítés az appba

GitHubon (böngészőből is megy): **Add file → Upload files** a
`app/src/main/assets/reranker/` mappába, mindkét fájlt, commit a `main`-re.
Utána futtasd az Android CI-t — az új APK-ban a reranker magától életre kel
(kódmódosítás nem kell). Ha kész, küldd el a 7) cella eval-számait a
kalibrációhoz!